# Adaptive Visual Retrieval — v2
## Region-Level Feedback-Adaptive Multimodal Retrieval

This notebook implements and evaluates the research question in
`research_proposal.md`:

> Does decomposing an image into detected object regions, and applying
> relevance feedback at the region level, converge to the user's intended
> result in fewer rounds than whole-image feedback — specifically on
> cluttered, multi-object images?

Structure mirrors the ablation table directly: four conditions, run and
evaluated side by side, rather than building only the final proposed system.


## 0. Setup

In [ ]:
# !pip install open_clip_torch ultralytics matplotlib scikit-learn --quiet

import torch
import open_clip
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import json
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 1. Load models

Reuses the v1 CLIP backbone (OpenCLIP ViT-B/32) for a fair baseline
comparison, plus a detector for the region-level conditions.


In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

# TODO: swap for a stronger open-vocab detector (e.g. OWL-ViT) if YOLOv8's
# fixed classes are too restrictive for the query set's object categories.
detector = YOLO("yolov8n.pt")


## 2. Query sets

Two labeled query sets are required (see `dataset.md` for the spec):
- `queries_simple.json` — single-subject images
- `queries_cluttered.json` — multi-object images

Each entry: `{"query": str | image_path, "query_type": "text"|"image",
"relevant_image_ids": [...]}`.

**TODO:** curate ~50 queries per set before running the ablation below.


In [ ]:
def load_query_set(path):
    with open(path) as f:
        return json.load(f)

# queries_simple = load_query_set("data/queries_simple.json")
# queries_cluttered = load_query_set("data/queries_cluttered.json")


## 3. Detection → region extraction

In [ ]:
def extract_regions(image_path, conf_threshold=0.25):
    """Run detector, return list of (crop: PIL.Image, bbox, conf) per image.
    Falls back to the full image as a single 'region' if nothing is detected,
    so downstream code doesn't need special-casing.
    """
    results = detector(image_path, conf=conf_threshold, verbose=False)[0]
    img = Image.open(image_path).convert("RGB")
    regions = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        regions.append((img.crop((x1, y1, x2, y2)), (x1, y1, x2, y2), conf))
    if not regions:
        regions = [(img, (0, 0, img.width, img.height), 1.0)]
    return regions


## 4. Multimodal embedding (whole-image and region-level)

In [ ]:
@torch.no_grad()
def embed_image(pil_image):
    tensor = clip_preprocess(pil_image).unsqueeze(0).to(device)
    feat = clip_model.encode_image(tensor)
    return (feat / feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

@torch.no_grad()
def embed_text(text):
    tokens = tokenizer([text]).to(device)
    feat = clip_model.encode_text(tokens)
    return (feat / feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

def embed_regions(image_path):
    """Returns list of (embedding, bbox, conf) for every detected region."""
    regions = extract_regions(image_path)
    return [(embed_image(crop), bbox, conf) for crop, bbox, conf in regions]


## 5. Feedback-adaptive re-ranking (Rocchio-style)

Same update rule for both whole-image and region-level conditions — the
difference is *what* the feedback is attached to (one embedding per image vs.
one embedding per region).


In [ ]:
def rocchio_update(query_vec, relevant_vecs, irrelevant_vecs, alpha=1.0, beta=0.75, gamma=0.15):
    query_vec = np.array(query_vec)
    pos = np.mean(relevant_vecs, axis=0) if relevant_vecs else np.zeros_like(query_vec)
    neg = np.mean(irrelevant_vecs, axis=0) if irrelevant_vecs else np.zeros_like(query_vec)
    updated = alpha * query_vec + beta * pos - gamma * neg
    return updated / np.linalg.norm(updated)


## 6. Ablation conditions

Each function returns a ranked list of image ids per round, so recall@k can
be computed per round downstream. Implementations are stubs — fill in once
the query sets and embedding cache exist.

| Condition | Function |
|---|---|
| Whole-image, no feedback | `run_whole_image_baseline` |
| Whole-image + feedback | `run_whole_image_feedback` |
| Region-level, no feedback | `run_region_baseline` |
| Region-level + feedback | `run_region_feedback` |


In [ ]:
def run_whole_image_baseline(query, image_embeddings, top_k=10):
    """query: text or image embedding. image_embeddings: {image_id: vec}"""
    # TODO: cosine similarity ranking, no feedback rounds — this is the v1 baseline.
    pass

def run_whole_image_feedback(query, image_embeddings, relevance_fn, n_rounds=5, top_k=10):
    # TODO: retrieve -> get feedback via relevance_fn -> rocchio_update -> repeat.
    pass

def run_region_baseline(query, region_embeddings, top_k=10):
    """region_embeddings: {image_id: [(vec, bbox, conf), ...]}. Aggregate
    region scores per image (e.g. max over regions) to produce a ranking."""
    pass

def run_region_feedback(query, region_embeddings, relevance_fn, n_rounds=5, top_k=10):
    # TODO: same loop as whole-image feedback, but feedback is attached to
    # individual regions rather than whole images.
    pass


## 7. Evaluation: recall@k per round

The central plot: recall@k on the y-axis, feedback round on the x-axis, one
line per condition, computed separately for `queries_simple` and
`queries_cluttered`. Convergence speed (rounds to reach some recall
threshold) is the key comparison, not final-round recall alone.


In [ ]:
def recall_at_k(ranked_ids, relevant_ids, k):
    return len(set(ranked_ids[:k]) & set(relevant_ids)) / max(len(relevant_ids), 1)

def plot_recall_per_round(results_by_condition, title):
    """results_by_condition: {condition_name: [recall_round_0, recall_round_1, ...]}"""
    plt.figure(figsize=(6, 4))
    for name, recalls in results_by_condition.items():
        plt.plot(range(len(recalls)), recalls, marker="o", label=name)
    plt.xlabel("Feedback round")
    plt.ylabel("Recall@k")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

# TODO: run all four conditions over queries_simple and queries_cluttered,
# collect recall-per-round, call plot_recall_per_round twice (once per set).


## 8. Notes / next steps

- [ ] Curate `queries_simple.json` and `queries_cluttered.json` (see `dataset.md`)
- [ ] Implement the four `run_*` functions
- [ ] Cache embeddings to disk (mirror v1's `.pt` cache pattern) to avoid recomputation
- [ ] Run ablation, generate recall-per-round plots for both query sets
- [ ] Write up findings in `research_proposal.md` under a new "Results" section
